# vLLM CoT Inference

Standalone notebook wrapper for `vllm_cot.py`. It does not import the `homework` package, so it can still run if `homework3/homework/` is removed. Keep this notebook next to `vllm_cot.py` and the `data/` directory.

This requires a Linux/CUDA environment with vLLM installed.

In [ ]:
# Optional install cell. Run only in an environment where vLLM is supported.
# %pip install vllm tqdm

In [ ]:
from pathlib import Path
import importlib.util

def find_project_file(filename: str) -> Path:
    candidates = [Path.cwd() / filename, Path.cwd() / "homework3" / filename]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {filename}. Run this notebook from homework3/ or the repo root.")

module_path = find_project_file("vllm_cot.py")
spec = importlib.util.spec_from_file_location("vllm_cot_standalone", module_path)
vc = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(vc)

data_dir = module_path.parent / "data"
module_path, data_dir

## Configuration

In [ ]:
checkpoint = "HuggingFaceTB/SmolLM2-360M-Instruct"
tensor_parallel_size = 1
dtype = "auto"
gpu_memory_utilization = 0.9
max_model_len = None
max_num_seqs = None

In [ ]:
model = vc.VLLMCoTModel(
    checkpoint=checkpoint,
    tensor_parallel_size=tensor_parallel_size,
    dtype=dtype,
    gpu_memory_utilization=gpu_memory_utilization,
    max_model_len=max_model_len,
    max_num_seqs=max_num_seqs,
)

## Single Question

In [ ]:
question = "How many gram are there per 6 kg?"
generation = model.generate(question)
parsed_answer = model.parse_answer(generation)
print(generation)
print({"parsed_answer": parsed_answer})

## Benchmark

In [ ]:
valid = vc.Dataset("valid", data_dir)
result = vc.benchmark(model, valid, max_question=100)
result

## Generate RFT CoT Data

This cell can be slow because it samples multiple completions for missed questions.

In [ ]:
run_datagen = False

if run_datagen:
    import argparse

    args = argparse.Namespace(
        checkpoint=checkpoint,
        tensor_parallel_size=tensor_parallel_size,
        dtype=dtype,
        gpu_memory_utilization=gpu_memory_utilization,
        max_model_len=max_model_len,
        max_num_seqs=max_num_seqs,
        split="train",
        data_dir=str(data_dir),
        output_json=str(data_dir / "rft_vllm.json"),
        max_samples=None,
        oversample=10,
        temperature=0.8,
        max_tokens=75,
    )
    vc.cmd_generate_dataset(args)